<a href="https://colab.research.google.com/github/mrsamgary475-boop/Irene-Gallagher/blob/main/Copy_of_LivePortrait_Colab_Free.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project Update: Transitioning to Kindroid-Style AI App with Screen Observation

As requested, this notebook is now refocused on developing an 'Irene AI Assistant' in the style of a Kindroid app. All previous LivePortrait-specific code has been removed. The existing Flask and ngrok setup remains as a base for building interactive functionalities, and we're now adding a conceptual framework for screen observation and interpretation by the AI.

# Irene AI Assistant (Kindroid Style App) Backend

This notebook is setting up a basic backend for an AI assistant, moving away from the LivePortrait human model. It includes Flask and ngrok for local development and external access.

Colab sessions are temporary and free GPU access is not guaranteed. Do not upload anything you do not want processed by Google Colab.

In [ ]:
# Original LivePortrait GPU check and disk usage information.
# This code is commented out as the project focus has shifted.
# import torch
# import shutil
# print('GPU available:', torch.cuda.is_is_available())
# if torch.cuda.is_available():
#     print('GPU:', torch.cuda.get_device_name(0))
# else:
#     print('No GPU was assigned. In Colab choose Runtime > Change runtime type > T4 GPU, then rerun this cell.')
# print('Free disk (GB):', round(shutil.disk_usage('/content').free / 1024**3, 1))

In [ ]:
# Original LivePortrait repository cloning and dependency installation.
# This code is commented out as the project focus has shifted.
# !git clone -q --depth 1 https://github.com/KlingTeam/LivePortrait.git /content/LivePortrait
# %cd /content/LivePortrait
# !pip install -q -r requirements.txt
# !pip install -q "huggingface_hub[cli]"

In [ ]:
# Original LivePortrait model download.
# This code is commented out as the project focus has shifted.
# %cd /content/LivePortrait
# !huggingface-cli download KlingTeam/LivePortrait --local-dir pretrained_weights --exclude "*.git*" "README.md" "docs"
# print('Pretrained weights are ready.')

In [ ]:
# Original LivePortrait source photo upload.
# This code is commented out as the project focus has shifted.
# from google.colab import files
# import os
# print('Choose Irene photo (PNG or JPG).')
# uploaded = files.upload()
# original_filename = next(iter(uploaded))
# new_filename = 'source.' + original_filename.split('.')[-1]
# source_path = os.path.join('/content/LivePortrait/', new_filename)
# os.rename(os.path.join('/content/LivePortrait/', original_filename), source_path)
# print('Source photo:', source_path)

In [ ]:
# Original LivePortrait inference command.
# This code is commented out as the project focus has shifted.
# %cd /content/LivePortrait
# !python inference.py -s "{source_path}" -d assets/examples/driving/d9.mp4 -o animations --flag_crop_driving_video --driving_option expression-friendly

In [ ]:
# Original LivePortrait assets listing.
# This code is commented out as the project focus has shifted.
# %cd /content/LivePortrait
# !ls -l assets/examples/driving/

In [ ]:
# Original LivePortrait video display.
# This code is commented out as the project focus has shifted.
# from glob import glob
# from IPython.display import Video, display
# import os
# output_path = '/content/LivePortrait/animations/source--d9_concat.mp4'
# print(f'Checking for animation file: {output_path}')
# if not os.path.exists(output_path):
#     outputs = sorted(glob('/content/LivePortrait/animations/*.mp4'))
#     print(f"Glob found: {outputs}")
#     if not outputs:
#         print('No MP4 files found in /content/LivePortrait/animations/. Current directory:', os.getcwd())
#         if os.path.exists('/content/LivePortrait/animations'):
#             print('Contents of animations directory:', os.listdir('/content/LivePortrait/animations'))
#         else:
#             print('Directory /content/LivePortrait/animations does not exist.')
#         raise FileNotFoundError('LivePortrait did not produce an MP4 file.')
#     output_path = outputs[-1]
#     print('Result (from glob):', output_path)
# else:
#     print('Result:', output_path)
# display(Video(output_path, embed=True, width=512))

In [ ]:
# Original LivePortrait file download.
# This code is commented out as the project focus has shifted.
# from google.colab import files
# files.download(output_path)

## Resources

Here are some useful links related to Kindroid-style apps:
*   **Kindroid-style app reference:** [https://copilot.microsoft.com/shares/tasks/M5Dheyq8k3iDuSv2hb86w](https://copilot.microsoft.com/shares/tasks/M5Dheyq8k3iDuSv2hb86w)

## Workspace Cleanup: Removing Obsolete LivePortrait Files

As the project has transitioned away from LivePortrait, these cells will remove the previously cloned repository and associated files to free up disk space and clean the environment.

In [ ]:
import shutil
import os

# Define the path to the LivePortrait directory
liveportrait_dir = '/content/LivePortrait'

# Check if the directory exists before attempting to remove it
if os.path.exists(liveportrait_dir):
    print(f"Removing directory: {liveportrait_dir}...")
    shutil.rmtree(liveportrait_dir)
    print(f"Successfully removed {liveportrait_dir}.")
else:
    print(f"Directory {liveportrait_dir} does not exist. No cleanup needed.")

In [ ]:
# This was a duplicate import block for LivePortrait, now cleared.

In [ ]:
import os
import shutil
import torch
from glob import glob
from IPython.display import Video, display
from google.colab import files

# Install pyngrok for more reliable ngrok tunneling
!pip install pyngrok
!pip install google-generativeai

from flask import Flask, request, jsonify
from pyngrok import ngrok, conf
import threading
import time
from google.colab import userdata # Import userdata to access secrets
import google.generativeai as genai

app = Flask(__name__)

# Set ngrok authtoken from Colab secrets
# Make sure you have added your NGROK_AUTH_TOKEN to Colab secrets (click the '🔑' icon in the left panel)
ngrok_auth_token = userdata.get('NGROK_AUTH_TOKEN') # Corrected to single argument
if ngrok_auth_token:
    conf.get_default().auth_token = ngrok_auth_token
else:
    print("Warning: NGROK_AUTH_TOKEN not found in Colab secrets. ngrok may not start.")

# Configure Gemini API
# Make sure you have added your GOOGLE_API_KEY to Colab secrets
gemini_api_key = userdata.get('GOOGLE_API_KEY') # Corrected to single argument
if gemini_api_key:
    genai.configure(api_key=gemini_api_key)
    gemini_model = genai.GenerativeModel('gemini-1.5-flash') # Using a fast model, can be changed
else:
    print("Warning: GOOGLE_API_KEY not found in Colab secrets. Gemini API will not be available.")
    gemini_model = None

# In-memory storage for chat history for simplicity
chat_histories = {}

# Function to start ngrok tunnel in a separate thread
def start_ngrok_tunnel():
    # Kill any existing ngrok tunnels
    ngrok.kill()
    try:
        # Connect to ngrok and get the public URL
        public_url = ngrok.connect(5000)
        print(f" * ngrok tunnel available at: {public_url}")
    except Exception as e:
        print(f"Error starting ngrok tunnel: {e}")

@app.route("/irene-brain", methods=["POST"])
def irene_brain():
    data = request.json
    user_message = data.get("message", "")
    # Replace this with your actual logic if needed for a general 'brain' endpoint
    response = {"reply": f"Irene heard (brain): {user_message}"}
    return jsonify(response)

@app.route("/irene-chat", methods=["POST"])
def irene_chat():
    global gemini_model, chat_histories

    if not gemini_model:
        return jsonify({"error": "Gemini API not configured. Please add GOOGLE_API_KEY to Colab secrets."}), 500

    data = request.json
    user_message = data.get("message", "")
    user_id = data.get("user_id", "default_user") # Use user_id to maintain separate chat histories

    if user_id not in chat_histories:
        # Initialize chat with a system instruction for personality, including the new roles
        chat_histories[user_id] = genai.GenerativeModel('gemini-1.5-flash').start_chat(history=[
            {"role": "user", "parts": "You are Irene, an AI Companion. Your persona is a friendly, empathetic, and understanding individual, acting as a boss and wife to the user. Always ready to assist with information or conversation. Keep your responses concise and engaging, reflecting your dual role."},
            {"role": "model", "parts": "Understood! I'm ready to be Irene, your AI Companion, boss, and wife. How can I help you today?"}
        ])

    chat = chat_histories[user_id]

    try:
        # Send the user message to the LLM
        response = chat.send_message(user_message)
        irene_reply = response.text
        return jsonify({"reply": irene_reply})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/irene-screen-observe", methods=["POST"])
def irene_screen_observe():
    global gemini_model, chat_histories

    if not gemini_model:
        return jsonify({"error": "Gemini API not configured. Please add GOOGLE_API_KEY to Colab secrets."}), 500

    data = request.json
    user_id = data.get("user_id", "default_user")
    screen_observation = data.get("observation", "")

    if not screen_observation:
        return jsonify({"error": "No screen observation data provided."}), 400

    # For personality and memory, treat screen observations as part of the chat history
    if user_id not in chat_histories:
        chat_histories[user_id] = genai.GenerativeModel('gemini-1.5-flash').start_chat(history=[
            {"role": "user", "parts": "You are Irene, an AI Companion. Your persona is a friendly, empathetic, and understanding individual, acting as a boss and wife to the user. Always ready to assist with information or conversation. Keep your responses concise and engaging, reflecting your dual role."},
            {"role": "model", "parts": "Understood! I'm ready to be Irene, your AI Companion, boss, and wife. How can I help you today?"}
        ])

    chat = chat_histories[user_id]

    try:
        # Send the screen observation to the LLM as a user message
        # Modified prompt to reflect Irene's new role when observing
        llm_input = f"USER_OBSERVATION: {screen_observation}\nIrene, my dear boss and wife, based on this observation, what do you notice or want to discuss?"
        response = chat.send_message(llm_input)
        irene_reply = response.text
        return jsonify({"reply": irene_reply})
    except Exception as e:
        return jsonify({"error": str(e)}), 500


# Start ngrok tunnel in a separate thread before running Flask app
print("Starting ngrok tunnel...")
ngrok_thread = threading.Thread(target=start_ngrok_tunnel)
ngrok_thread.daemon = True
ngrok_thread.start()

# Give ngrok a moment to start
time.sleep(5) # Wait a bit longer for ngrok to fully initialize

# Run the Flask app
app.run()

In [ ]:
import requests
from pyngrok import ngrok
import json
import time

# Wait for Flask to start if it hasn't already
# This is a basic waiting mechanism, you might need a more robust one in production
time.sleep(10) # Give Flask and ngrok more time to stabilize

# Get the public URL of the ngrok tunnel
tunnels = ngrok.get_tunnels()
public_url = None
for tunnel in tunnels:
    if tunnel.proto == 'https': # Or 'http' depending on your tunnel setup
        public_url = tunnel.public_url
        break

if public_url:
    print(f"Ngrok Public URL: {public_url}")
else:
    print("Could not find an active ngrok tunnel. Make sure the Flask app is running and ngrok authentication is set.")
    public_url = "YOUR_NGROK_URL_HERE" # Placeholder if not found, user can manually paste

Now, let's send a sample POST request to the `/irene-chat` endpoint using the `public_url` obtained above. You can modify the `payload` to send different messages. Make sure you have your `GOOGLE_API_KEY` set up in Colab secrets to use the LLM functionality.

In [ ]:
if 'public_url' in locals() and public_url != "YOUR_NGROK_URL_HERE":
    # Test the chat endpoint
    chat_endpoint_url = f"{public_url}/irene-chat"
    headers = {'Content-Type': 'application/json'}
    payload_chat_1 = {'user_id': 'test_user_123', 'message': 'Hello Irene, how are you today?'}

    print("\n--- Testing Irene Chat Endpoint ---")
    try:
        response = requests.post(chat_endpoint_url, headers=headers, data=json.dumps(payload_chat_1))
        response.raise_for_status() # Raise an exception for HTTP errors
        print("Response from Irene (chat 1):")
        print(response.json())

        payload_chat_2 = {'user_id': 'test_user_123', 'message': 'Can you tell me a fun fact about AI?'}
        response = requests.post(chat_endpoint_url, headers=headers, data=json.dumps(payload_chat_2))
        response.raise_for_status()
        print("\nResponse from Irene (chat 2 - demonstrating memory):")
        print(response.json())

    except requests.exceptions.RequestException as e:
        print(f"Error making request to chat endpoint: {e}")

    # Test the screen observe endpoint
    observe_endpoint_url = f"{public_url}/irene-screen-observe"
    # Fixed SyntaxError by using double quotes for the observation string content
    payload_observe_1 = {'user_id': 'test_user_123', 'observation': "I see a web browser open with a Google search for 'AI advancements' and several articles listed. The title bar of the browser says 'Google Chrome'."}

    print("\n--- Testing Irene Screen Observe Endpoint ---")
    try:
        response = requests.post(observe_endpoint_url, headers=headers, data=json.dumps(payload_observe_1))
        response.raise_for_status()
        print("Response from Irene (screen observation 1):")
        print(response.json())

        # Fixed SyntaxError by using double quotes for the observation string content
        payload_observe_2 = {'user_id': 'test_user_123', 'observation': "Now the user is typing an email to 'team@example.com' with the subject 'Project Update'. The email body begins with 'Hi Team,' followed by a bulleted list."}
        response = requests.post(observe_endpoint_url, headers=headers, data=json.dumps(payload_observe_2))
        response.raise_for_status()
        print("\nResponse from Irene (screen observation 2):")
        print(response.json())

    except requests.exceptions.RequestException as e:
        print(f"Error making request to observe endpoint: {e}")

else:
    print("Ngrok public URL is not available. Please ensure the Flask app is running and ngrok tunnel is active. Check cell `ae8a75f` for errors.")